# പാഠം 01 - എഐ ഏജന്റുകളിലേക്ക് പരിചയം

**എഐ ഏജന്റുകൾ ആരംഭിക്കാം** കോഴ്സിലെ ആദ്യ പാഠത്തിലേക്ക് സ്വാഗതം!

ഒരു **എഐ ഏജന്റ്** വലിയ ഭാഷ മോഡൽ (LLM) ഒരു തർക്ക എഞ്ചിനായി ഉപയോഗിച്ച് യാഥാർത്ഥ്യ ലോകത്തിൽ *പ്രവർത്തനങ്ങൾ* നിർവഹിക്കുന്ന ഒരു പ്രോഗ്രാമാണ് — API കോളുചെയ്യൽ, ഡാറ്റാബേസുകൾ ചോദ്യംചെയ്യൽ, അല്ലെങ്കിൽ കോഡ് പ്രവർത്തിപ്പിക്കൽ — ഉപഭോക്താവിനായി ഒരു ലക്ഷ്യം നേടാനായി.

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ നിങ്ങളുടെ ആദ്യത്തെ ഏജന്റ് നിർമ്മിക്കും: അവധിക്കാല ഗമനസ്ഥലങ്ങൾ ശുപാർശ ചെയ്യുന്ന ഒരു **ട്രാവൽ ഏജന്റ്**. ഇതിനു വേണ്ടി നിങ്ങൾക്ക് പഠിക്കാനുണ്ടാകുന്നത്:

1. **Microsoft Agent Framework** ഉപയോഗിച്ച് Microsoft Foundry Agent Service-ൽ കണക്ട് ചെയ്യുന്നത്.
2. ഏജന്റിന് ഒരു **ടൂൾ** നൽകുന്നത് — ഏജന്റ് വിളിക്കാവുന്ന സാമാന്യ പൈത്തൺ ഫങ്‌ഷൻ.
3. ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ പ്രതികരണം പരിശോധന നടത്തുക.
4. ഏജന്റിന്റെ പ്രതികരണത്തെ ടോക്കൺ-ടോക്കൺ സ്ട്രീം ചെയ്യുക.


## സെറ്റപ്പ്

ഈ നോട്ട്‌ബുക് چلിയ്ക്കുന്നതിന് മുമ്പ്, ദയവായി നിങ്ങൾക്കുണ്ടെന്ന് ഉറപ്പാക്കുക:

1. **പ്രവർത്തന ഏർപ്പെടുത്തിയ ഒരു Microsoft Foundry പ്രോജക്ട്** (ഉദാഹരണത്തിന് `gpt-5-mini` പോലെ ഒരു ചാറ്റ് മോഡൽ).
2. **Azure CLI ഉപയോഗിച്ച് ലോഗിൻ ചെയ്തിരിക്കുന്നത്** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
3. **ആവശ്യമായ എൻവയോൺമെന്റ് വേരിയബിളുകൾ ക്രമീകരിക്കുക:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്ടിന്റെ എൻഡ്പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങൾ ഏർപ്പെടുത്തിയ മോഡലിന്റെ പേര്.

താഴെ കൊടുത്ത സെൽ നിങ്ങൾക്ക് ആവശ്യമായ Python പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യും.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## നിങ്ങളുടെ ആദ്യ ഏജന്റ് സൃഷ്ടിക്കൽ

ഒരു ഏജന്റിന് രണ്ടു കാര്യങ്ങൾ ആവശ്യമാണ്:

- **നിർദേശംകൾ** അത് *ആരാണ്* എന്നും *എങ്ങനെ പെരുമാറണം* എന്നും പറയുന്നത് (ഒരു സിസ്റ്റം പ്രോമ്പ്റ്റ്).
- **ഉപകരണങ്ങൾ** — `@tool` ഉപയോഗിച്ച് അലങ്കരിച്ച_python ഫംഗ്ഷനുകൾ_, ഏജന്റ് വിവരങ്ങൾ നേടുന്നതിനോ പ്രവർത്തികള്‍ ചെയ്യുന്നതിനോ വിളിക്കാവുന്നവ.

താഴെ നമ്മൾ ഒരു ലളിതമായ ഉപകരണം നിർവചിക്കുന്നു, അത് പ്രചാരത്തിലുള്ള അവധി ഗമന സ്ഥലങ്ങളുടെ പട്ടിക നൽകുന്നു. ഉപയോക്താവ് യാത്രാ ശുപാർശകൾ ചോദിക്കുമ്പോൾ ഏജന്റ് ഈ ഉപകരണം ഉപയോഗിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## സ്ട്രീമിംഗ് മറുപടികൾ

കൂടുതല്‍ ഇന്ററാക്ടിവ് അനുഭവത്തിനായി, ഏജന്റിന്റെ മറുപടി **സ്ട്രീം** ചെയ്യാൻ കഴിയും. പൂർണ്ണ മറുപടി ലഭിക്കാൻ കാത്തിരിക്കുന്ന പകരം, ഏജന്റ് സൃഷ്ടിക്കുന്നതുപോലെ പിഞ്ചു പിഞ്ചായി വാചകങ്ങൾ സാധിക്കുന്നു. വിശേഷിച്ച് സജീവമായി ഫലങ്ങൾ കാണിക്കാൻ ചാറ്റ് ഇൻറർഫേസുകളിൽ ഇത് വളരെ പ്രയോജനം ആണ്.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

- **Microsoft Foundry Agent Service-നെ `FoundryChatClient` വഴി ബന്ധിപ്പിക്കുന്ന ഒരു പ്രൊവൈഡർ സൃഷ്ടിക്കുക**.
- **ഏജന്റ് നിങ്ങളുടെ Python ഫംഗ്ഷനുകൾ വിളിക്കാൻ സാധിക്കുന്ന രീതിയിൽ `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ഒരു ടൂൾ നിർവ്വചിക്കുക**.
- **ഉപയോക്തൃ സന്ദേശത്തോടൊപ്പം ഏജന്റ് പ്രവർത്തിപ്പിച്ച് അതിന്റെ പ്രതികരണം പ്രിൻറ് ചെയ്യുക**.
- **യഥാർത്ഥസമയ ഔട്ട്പുട്ടിനായി പ്രതികരണങ്ങൾ സ്ട്രീം ചെയ്ത് കാണിക്കുക**.

അടുത്ത പാഠത്തിൽ ഏജന്റിക് ഫെയിമ്വർക്കുകൾ കൂടുതൽ ആഴത്തിൽ പരിശോധിക്കുകയും ഏജന്റുകൾക്ക് ശക്തമായ ടൂളുകളും ബഹു-പടി വിവേകന ശേഷികളും നൽകാൻ പഠിക്കുകയും ചെയ്യും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
